### Limpeza da base de dados 


Carregamento dos dados 

In [1]:
from pathlib import Path
import pandas as pd

base_path = Path.cwd().parent  
file_path = base_path / "data" / "raw" / "Base_de_Dados_IDSC-BR_2025.xlsx"

df = pd.read_excel(file_path, sheet_name="Todos os Dados")

df.head()

,ID,Município,População_2022,UF,Pontuação Indice ODS 2025,Classificação 2025,Valores faltantes,Porcentagem valores faltantes,Goal 1 Score,Goal 2 Score,...,Normalizado (0-100): SDG16_TRANSP,Painel: SDG16_TRANSP,SDG17_1_INVST_PB,year_SDG17_1_INVST_PB,Normalizado (0-100): SDG17_1_INVST_PB,Painel: SDG17_1_INVST_PB,SDG17_3_P_RC_TRB,year_SDG17_3_P_RC_TRB,Normalizado (0-100): SDG17_3_P_RC_TRB,Painel: SDG17_3_P_RC_TRB
0,1100015,Alta Floresta D'Oeste,21495,RO,41.885064,5182,4,4.0,53.769451,53.255278,...,66.667,yellow,13.216,2023.0,0.000000,red,5.257,2023.0,8.108054,orange
1,1100023,Ariquemes,96833,RO,45.475390,4448,1,1.0,54.219225,50.890756,...,66.667,yellow,85.642,2023.0,1.133196,red,18.394,2023.0,34.298246,yellow
2,1100031,Cabixi,5363,RO,47.569518,3758,6,6.0,49.228293,43.817398,...,66.667,yellow,909.852,2023.0,38.715329,green,5.686,2023.0,8.963317,orange
3,1100049,Cacoal,86895,RO,48.582323,3366,0,0.0,50.752090,51.083977,...,50.000,yellow,328.073,2023.0,12.187507,orange,21.627,2023.0,40.743620,green
4,1100056,Cerejeiras,15890,RO,48.718897,3312,2,2.0,56.552994,50.268036,...,66.667,yellow,246.352,2023.0,8.461212,orange,12.178,2023.0,21.905901,yellow


Identificação e remoção dos metadados 

In [2]:

df_info = df[['ID', 'Município', 'UF', 'População_2022', 'Pontuação Indice ODS 2025']].copy()

colunas_remover = [c for c in df.columns if c.startswith('year_') or c.startswith('Painel: ')]
df = df.drop(columns=colunas_remover)

colunas_brutas = [c for c in df.columns if str(c).startswith('SDG') and 'Normalizado' not in str(c)]
df = df.drop(columns=colunas_brutas)

Remoção de colunas categóricas e descritivas

`ID` é preservado e vira o índice `id_municipio` da base, em vez de ser descartado — os notebooks seguintes (clustering, predição por cluster, atribuição por centróide) dependem desse identificador único de município como índice.

In [3]:
colunas_ignorar = ['Município', 'UF', 'Classificação 2025', 'Valores faltantes', 'Porcentagem valores faltantes']
df = df.drop(columns=[c for c in colunas_ignorar if c in df.columns])

df = df.rename(columns={'ID': 'id_municipio'}).set_index('id_municipio')

Remoção de colunas que podem ocasionar vazamento de dados 

In [4]:
colunas_vazamento = [
    'Goal 4 Score', 
    'ODS4_reg'
]
df = df.drop(columns=[c for c in colunas_vazamento if c in df.columns])

Salvamento da base de dados processada (completa, antes do split)

In [5]:

file_path = base_path / "data" / "processed" / "Base_de_Dados_Processada.csv"

df.to_csv(file_path, index_label="id_municipio")